# 자율주행컴퓨팅 실습 2 — LiDAR 3D Object Detection 해부하기 (VoxelNet 중심)

**Part A. Point cloud 라는 데이터 타입** — 실제 KITTI 프레임 1장으로
- `.bin` 파일 안에 무엇이 들어 있는가 (N × 4 float)
- 3D / BEV / range image / 카메라 투영 — 같은 데이터를 보는 네 가지 방법
- 수업에서 다룬 LiDAR 기반 방법 **PIXOR · VoxelNet · PointRCNN** 은 이 데이터를 각각 어떤 입력으로 바꾸는가
- 왜 거리가 멀수록 물체가 "사라지는가" (sparsity)

**Part B. VoxelNet 을 처음부터 끝까지** — 순수 PyTorch 로 구현한 VoxelNet 에 point cloud 를 넣고 단계별 텐서를 꺼내 봅니다.

| 단계 | 질문 | 섹션 |
|---|---|---|
| Voxel partition & grouping | 불규칙한 점 집합을 어떻게 3D 격자로 묶는가? | B1 |
| VFE (Voxel Feature Encoding) | voxel 안의 점들(개수 제각각)로부터 feature 를 어떻게 **학습**하는가? | B2, B3 |
| 3D conv middle layers | 3D 격자가 어떻게 2D BEV feature map 이 되는가? | B3 |
| RPN · 3D anchor · target | 실습 1의 anchor 가 3D 에서는 어떻게 바뀌는가? | B4 |
| Loss · 후처리 | 수만 개 negative vs 수십 개 positive — VoxelNet 은 불균형을 어떻게 다루는가? | B5 |
| 학습 전 vs 후 | 같은 point cloud 에 대해 무엇이 달라지는가? | B6, B7 |
| 실제 데이터 | 합성 데이터로 학습한 모델은 실제 KITTI 에서도 동작할까? | B8 |

**학습 데이터에 대해** — KITTI 3D 는 개별 등록이 필요하고 용량(약 30 GB)과 학습 시간(수 시간)이 커서 수업 중 학습이 어렵습니다.
그래서 Part B 의 학습은 **LiDAR ray-casting 시뮬레이터로 만든 합성 장면**으로 합니다. KITTI 의 HDL-64E 와 같은 64 채널 / 장착 높이 1.73 m 로 레이저를 쏘아,
가림(occlusion)과 거리별 희소성이 실제와 같은 원리로 생깁니다.

> ⚙️ Colab 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 선택한 뒤 시작하세요.

## 0. 환경 설정

In [ ]:
import os, math, random, time, pickle, urllib.request
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from torchvision.ops import box_iou, batched_nms, sigmoid_focal_loss
try:
    import plotly.graph_objects as go
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"], check=True)
    import plotly.graph_objects as go

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 0
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all()
print("torch", torch.__version__, "| device:", device)
if device.type == "cpu":
    print("⚠️ GPU 가 잡히지 않았습니다. 런타임 유형을 T4 GPU 로 바꿔주세요.")

### 실습 설정
- Part A 는 공개 저장소의 KITTI 샘플 프레임 1장을 자동으로 받아 사용합니다.
- Part B 는 시뮬레이터로 만든 합성 장면으로 VoxelNet 을 `EPOCHS` 만큼 직접 학습합니다.

In [ ]:
DATA_ROOT   = "/content/data"

# ---- VoxelNet 설정 (논문의 KITTI Car 설정에서 범위와 채널 수만 축소) ----
POINT_RANGE = [0.0, -25.6, -3.0, 51.2, 25.6, 1.0]     # x_min, y_min, z_min, x_max, y_max, z_max (m)
VX, VY, VZ  = 0.2, 0.2, 0.4                           # voxel 크기 (m) — 논문과 동일 → 256 x 256 x 10 격자
VOXEL       = VX
MAX_PTS     = 32                                      # voxel 당 최대 점 수 (논문 T = 35, 넘으면 무작위 샘플링)
MAX_VOXELS  = 20000                                   # 장면당 최대 (비어있지 않은) voxel 수
USE_INTENSITY = False   # 논문은 반사강도 r 을 쓰지만, 합성 데이터의 r 은 실제와 분포가 달라 기본으로 0 처리 (B8 의 실제 데이터 적용을 위해)

CLASSES      = ["Car", "Pedestrian"]
ANCHOR_SIZE  = {"Car": (3.9, 1.6, 1.56), "Pedestrian": (0.8, 0.6, 1.73)}   # (l, w, h)
ANCHOR_Z     = {"Car": -0.95, "Pedestrian": -0.865}                        # 박스 중심 z (센서 기준, 지면 = -1.73)
IOU_THR      = {"Car": (0.6, 0.45), "Pedestrian": (0.5, 0.35)}             # (positive ≥, negative <)  BEV IoU
LOSS_ALPHA, LOSS_BETA = 1.5, 1.0                                            # VoxelNet 손실의 positive / negative 가중치
CLS_LOSS     = "voxelnet"   # "voxelnet" (논문: positive/negative 를 각각 평균) 또는 "focal" (실습 1 Part 2 의 RetinaNet 방식)

N_TRAIN, N_VAL = 600, 60
EPOCHS      = 15
SNAP_EPOCHS = [0, 1, 3, 8, EPOCHS]   # 이 epoch 들의 가중치를 저장해 비교 (0 = 학습 전)
BATCH_SIZE  = 4
LR          = 2e-3

XMIN, YMIN, ZMIN, XMAX, YMAX, ZMAX = POINT_RANGE
NX, NY, NZ = int(round((XMAX - XMIN) / VX)), int(round((YMAX - YMIN) / VY)), int(round((ZMAX - ZMIN) / VZ))
print(f"voxel grid: {NX} x {NY} x {NZ} = {NX * NY * NZ:,} voxels (each {VX} x {VY} x {VZ} m)")

### 공통 helper: 3D 박스 표현 `(x, y, z, l, w, h, yaw)` — LiDAR 좌표계 (x 앞, y 왼쪽, z 위), (x, y, z) 는 박스 중심

In [ ]:
def box_corners_bev(b):                 # (B,7) → (B,4,2)
    b = torch.as_tensor(b, dtype=torch.float32).reshape(-1, 7)
    c, s = torch.cos(b[:, 6]), torch.sin(b[:, 6])
    dx = torch.tensor([0.5, 0.5, -0.5, -0.5]) [None] * b[:, 3:4]
    dy = torch.tensor([0.5, -0.5, -0.5, 0.5])[None] * b[:, 4:5]
    x = b[:, 0:1] + dx * c[:, None] - dy * s[:, None]
    y = b[:, 1:2] + dx * s[:, None] + dy * c[:, None]
    return torch.stack([x, y], -1)

def box_corners_3d(b):                  # (B,7) → (B,8,3)  아래 4개 + 위 4개
    b = torch.as_tensor(b, dtype=torch.float32).reshape(-1, 7)
    bev = box_corners_bev(b)
    zb, zt = (b[:, 2] - b[:, 5] / 2)[:, None, None], (b[:, 2] + b[:, 5] / 2)[:, None, None]
    return torch.cat([torch.cat([bev, zb.expand(-1, 4, 1)], -1), torch.cat([bev, zt.expand(-1, 4, 1)], -1)], 1)

def points_in_boxes(pts, boxes):        # (N,≥3), (B,7) → (N,B) bool
    pts, boxes = torch.as_tensor(pts)[:, :3].float(), torch.as_tensor(boxes).reshape(-1, 7).float()
    d = pts[:, None] - boxes[None, :, :3]
    c, s = torch.cos(boxes[:, 6]), torch.sin(boxes[:, 6])
    lx, ly = d[..., 0] * c + d[..., 1] * s, -d[..., 0] * s + d[..., 1] * c
    return (lx.abs() <= boxes[:, 3] / 2) & (ly.abs() <= boxes[:, 4] / 2) & (d[..., 2].abs() <= boxes[:, 5] / 2)

def bev_ax(ax, pts=None, title=None, s=0.3, color_by="z", full=True):
    if pts is not None:
        p = torch.as_tensor(pts).cpu()
        col = p[:, 2] if color_by == "z" else p[:, 3]
        ax.scatter(p[:, 0], p[:, 1], s=s, c=col, cmap="viridis", vmin=-2.2 if color_by == "z" else 0, vmax=1.0, linewidths=0)
    if full: ax.set_xlim(XMIN, XMAX); ax.set_ylim(YMIN, YMAX)
    ax.set_aspect("equal"); ax.set_facecolor("#111111")
    ax.set_xlabel("x (forward, m)"); ax.set_ylabel("y (left, m)")
    if title: ax.set_title(title, fontsize=10)

def draw_bev_boxes(ax, boxes, color="lime", lw=1.5, ls="-", texts=None, fs=7):
    boxes = torch.as_tensor(boxes).reshape(-1, 7).detach().cpu()
    if len(boxes) == 0: return
    if texts is not None and len(boxes) > 20: texts = None
    for k, (cn, b) in enumerate(zip(box_corners_bev(boxes), boxes)):
        c = color[k] if isinstance(color, list) else color
        ax.add_patch(patches.Polygon(cn.numpy(), closed=True, fill=False, edgecolor=c, lw=lw, ls=ls))
        f = (cn[0] + cn[1]) / 2                                   # 앞쪽 변의 중점 → 진행 방향 표시
        ax.plot([b[0], f[0]], [b[1], f[1]], color=c, lw=lw)
        if texts is not None: ax.text(float(b[0]), float(b[1]) + 1.2, texts[k], color=c, fontsize=fs, ha="center")

def show3d(fig): fig.show()

def plot3d(pts, groups=(), title="", max_pts=40000):
    p = torch.as_tensor(pts).cpu()
    if len(p) > max_pts: p = p[torch.randperm(len(p))[:max_pts]]
    data = [go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="markers", name="points",
                         marker=dict(size=1.3, color=p[:, 2], colorscale="Viridis", cmin=-2.2, cmax=1.0))]
    edges = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7)]
    for boxes, color, name in groups:
        xs, ys, zs = [], [], []
        for cn in box_corners_3d(torch.as_tensor(boxes).reshape(-1, 7).cpu()):
            for i, j in edges + [(0, 5), (1, 4)]:              # 앞면 X 표시 = 진행 방향
                xs += [cn[i, 0], cn[j, 0], None]; ys += [cn[i, 1], cn[j, 1], None]; zs += [cn[i, 2], cn[j, 2], None]
        data.append(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color=color, width=4), name=name))
    fig = go.Figure(data)
    fig.update_layout(title=title, height=600, margin=dict(l=0, r=0, t=40, b=0), paper_bgcolor="black",
                      font_color="white", scene=dict(aspectmode="data", bgcolor="black",
                      camera=dict(eye=dict(x=-0.9, y=-0.9, z=0.6))))
    show3d(fig)

# Part A. Point cloud 라는 데이터 타입

## A1. 실제 KITTI 프레임 불러오기
KITTI 의 LiDAR 파일 `velodyne/000008.bin` 은 헤더도 없는 **float32 의 나열**입니다. 4개씩 끊으면 한 점 = `(x, y, z, reflectance)`.
3D 박스 라벨은 **카메라 좌표계**로 저장되어 있어서, calibration 행렬로 LiDAR 좌표계로 바꿔야 합니다.

> 자동으로 받는 샘플 파일은 **카메라 시야(전방 약 90°) 안의 점만 남긴 버전**이라 원본(약 12만 점)보다 점이 적습니다.

In [ ]:
def cam_to_lidar_boxes(loc, lhw, ry, lidar2cam):
    # KITTI 라벨: loc = 박스 바닥 중심 (카메라 좌표, y 아래 방향), lhw = (l, h, w), ry = 카메라 y축 회전
    loc, lhw, ry = np.asarray(loc, float).reshape(-1, 3), np.asarray(lhw, float).reshape(-1, 3), np.asarray(ry, float).reshape(-1)
    ctr = loc.copy(); ctr[:, 1] -= lhw[:, 1] / 2                                   # 바닥 중심 → 박스 중심
    xyz = (np.linalg.inv(lidar2cam) @ np.c_[ctr, np.ones(len(ctr))].T).T[:, :3]
    yaw = -ry - np.pi / 2
    return np.c_[xyz, lhw[:, 0], lhw[:, 2], lhw[:, 1], yaw].astype(np.float32)        # (x,y,z,l,w,h,yaw)

def load_kitti_sample():
    d = os.path.join(DATA_ROOT, "kitti_sample"); os.makedirs(d, exist_ok=True)
    base = "https://raw.githubusercontent.com/open-mmlab/mmdetection3d/main/demo/data/kitti/"
    for f in ["000008.bin", "000008.png", "000008.pkl"]:
        if not os.path.exists(os.path.join(d, f)): urllib.request.urlretrieve(base + f, os.path.join(d, f))
    pts = np.fromfile(os.path.join(d, "000008.bin"), dtype=np.float32).reshape(-1, 4)
    info = pickle.load(open(os.path.join(d, "000008.pkl"), "rb"))["data_list"][0]
    l2c = np.array(info["images"]["CAM2"]["lidar2cam"]); l2i = np.array(info["images"]["CAM2"]["lidar2img"])
    names = {0: "Pedestrian", 1: "Cyclist", 2: "Car"}
    inst = [x for x in info["instances"] if x["bbox_label_3d"] in names]
    b3 = np.array([x["bbox_3d"] for x in inst]).reshape(-1, 7)                      # (x,y,z, l,h,w, ry)
    boxes = cam_to_lidar_boxes(b3[:, :3], b3[:, 3:6], b3[:, 6], l2c)
    return dict(points=pts, boxes=boxes, names=[names[x["bbox_label_3d"]] for x in inst],
                image=np.array(Image.open(os.path.join(d, "000008.png")).convert("RGB")),
                lidar2img=l2i, path=os.path.join(d, "000008.bin"))

kitti = load_kitti_sample()
P = kitti["points"]
print(f"파일 크기 {os.path.getsize(kitti['path']):,} bytes = {len(P):,} points x 4 values x 4 bytes (float32)")
print("배열 shape:", P.shape, P.dtype)
print("처음 5개 점 (x, y, z, reflectance):\n", np.round(P[:5], 3))
for i, n in enumerate(["x", "y", "z", "reflectance"]):
    print(f"  {n:<12s} min {P[:, i].min():8.2f}  max {P[:, i].max():8.2f}  mean {P[:, i].mean():7.2f}")
pin = points_in_boxes(P, kitti["boxes"]).sum(0)
for n, b, k in zip(kitti["names"], kitti["boxes"], pin.tolist()):
    print(f"  GT {n:<10s} center ({b[0]:5.1f}, {b[1]:5.1f}, {b[2]:5.2f})  lwh ({b[3]:.2f}, {b[4]:.2f}, {b[5]:.2f})  "
          f"yaw {b[6]:5.2f}  거리 {np.hypot(b[0], b[1]):5.1f} m  내부 점 {k}")

> 🤔 **생각해보기** — 이미지와 비교해 point cloud 는 무엇이 다른가요?
> 프레임마다 점의 개수가 같은가요? 점의 순서를 바꾸면 다른 장면이 되나요? (바로 아래 셀에서 확인해 보세요.)

**답변**

이미지는 픽셀들이 일정한 높이×너비의 2차원 격자에 놓이지만, point cloud는 물체 표면에서 측정한 불규칙하고 희소한 3차원 점들의 집합이다. 이 KITTI 프레임은 17,238개의 점으로 이루어진 (17238, 4) 배열이며, 각 점은 (x, y, z, reflectance)를 갖는다. 프레임마다 거리, 가림, 반사 특성 및 측정 범위가 달라지므로 점의 개수도 항상 같지는 않다. 점의 행 순서를 바꿔도 좌표와 반사강도로 이루어진 점 집합은 그대로여서 같은 장면을 나타낸다. 실제 출력에서도 “순서를 섞은 뒤에도 같은 장면인가? True | 그러나 배열로는 다름: True”로 확인되었다.


In [ ]:
perm = np.random.permutation(len(P))
print("순서를 섞은 뒤에도 같은 장면인가?", np.allclose(np.sort(P[perm], axis=0), np.sort(P, axis=0)),
      "| 그러나 배열로는 다름:", not np.allclose(P[perm], P))
plot3d(P, [(kitti["boxes"], "lime", "GT boxes")], title="KITTI frame — drag to rotate (color = height)")

## A2. 같은 데이터를 보는 네 가지 방법 — 3D(위), BEV, Range image, 카메라 투영
- **BEV (Bird's Eye View)** : 위에서 내려다본 평면. 물체끼리 겹치지 않고 크기가 거리와 무관
- **Range image** : LiDAR 가 실제로 측정하는 방식 그대로 (가로 = 방위각, 세로 = 레이저 채널, 값 = 거리)
- **카메라 투영** : calibration 행렬로 점을 카메라 이미지 위에 올림 → 각 점이 실제 장면의 어디에 해당하는지 확인

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
bev_ax(ax, P, "BEV (color = height z)", s=0.6, full=False)
draw_bev_boxes(ax, kitti["boxes"], color="lime", texts=kitti["names"])
ax.plot(0, 0, "r^", ms=10); ax.text(0.5, -2, "LiDAR", color="r")
ax.set_xlim(-2, P[:, 0].max()); ax.set_ylim(P[:, 1].min(), P[:, 1].max()); plt.show()

# Range image: (elevation, azimuth) 격자에 거리 값을 채움
r = np.linalg.norm(P[:, :3], axis=1)
az = np.degrees(np.arctan2(P[:, 1], P[:, 0])); el = np.degrees(np.arcsin(P[:, 2] / r))
H, W = 64, 1024
row = np.clip(((2.0 - el) / (2.0 + 24.9) * (H - 1)).round().astype(int), 0, H - 1)
col = np.clip(((az - az.min()) / (az.max() - az.min()) * (W - 1)).round().astype(int), 0, W - 1)
rimg = np.zeros((H, W)); order = np.argsort(-r); rimg[row[order], col[order]] = r[order]   # 가까운 점이 우선
fig, ax = plt.subplots(figsize=(18, 3))
ax.imshow(rimg, cmap="turbo", aspect="auto", vmax=50)
ax.set_title(f"Range image ({H} lasers x {W} azimuth bins, value = range, black = no return)")
ax.set_xlabel(f"azimuth  ({az.max():.0f}° … {az.min():.0f}°)"); ax.set_ylabel("laser channel (up → down)")
plt.show()

In [ ]:
def project(pts, l2i):
    h = np.c_[pts[:, :3], np.ones(len(pts))] @ l2i.T
    return h[:, :2] / h[:, 2:3], h[:, 2]

im = kitti["image"]; uv, depth = project(P, kitti["lidar2img"])
ok = (depth > 0.5) & (uv[:, 0] >= 0) & (uv[:, 0] < im.shape[1]) & (uv[:, 1] >= 0) & (uv[:, 1] < im.shape[0])
fig, axes = plt.subplots(2, 1, figsize=(18, 11))
axes[0].imshow(im); axes[0].scatter(uv[ok, 0], uv[ok, 1], c=depth[ok], s=1.5, cmap="turbo", vmax=50)
axes[0].set_title(f"LiDAR points projected onto the camera image ({ok.sum():,} of {len(P):,} points visible)"); axes[0].axis("off")
axes[1].imshow(im); axes[1].axis("off"); axes[1].set_title("3D GT boxes projected onto the image")
edges = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7)]
for cn in box_corners_3d(kitti["boxes"]).numpy():
    q, dpt = project(cn, kitti["lidar2img"])
    if (dpt > 0).all():
        for i, j in edges: axes[1].plot(q[[i, j], 0], q[[i, j], 1], color="lime", lw=1.5)
plt.tight_layout(); plt.show()

## A3. 수업에서 다룬 LiDAR 기반 방법들은 이 데이터를 어떻게 "입력"으로 바꾸는가

| 논문 | 입력 표현 | 이후 처리 |
|---|---|---|
| **PIXOR** | BEV **점유 격자**: 높이 구간마다 "점이 있으면 1, 없으면 0" (사람이 정한 규칙) | 바로 2D CNN |
| **VoxelNet** | 3D voxel 격자 + **학습되는** voxel feature | 3D conv → 2D RPN — **Part B 에서 직접 구현** |
| **PointRCNN** | 원본 점 그대로 | 점마다 전경/배경을 구분 → 전경 점이 박스를 제안 (2-stage) |

### PIXOR : 높이 구간별 점유 격자
(PIXOR 원 논문은 0.1 m 간격 35개 구간을 쓰고, 여기서는 보기 쉽게 0.5 m 간격 7개 구간을 씁니다.)

In [ ]:
def pixor_occupancy(pts, res=0.1, z_edges=np.arange(-2.5, 1.01, 0.5)):
    p = pts[(pts[:, 0] >= XMIN) & (pts[:, 0] < XMAX) & (pts[:, 1] >= YMIN) & (pts[:, 1] < YMAX)
            & (pts[:, 2] >= z_edges[0]) & (pts[:, 2] < z_edges[-1])]
    Wb, Hb, S = int(round((XMAX - XMIN) / res)), int(round((YMAX - YMIN) / res)), len(z_edges) - 1
    ix = np.clip(((p[:, 0] - XMIN) / res).astype(int), 0, Wb - 1); iy = np.clip(((p[:, 1] - YMIN) / res).astype(int), 0, Hb - 1)
    iz = np.clip(np.searchsorted(z_edges, p[:, 2], side="right") - 1, 0, S - 1)
    occ = np.zeros((S, Hb, Wb)); occ[iz, iy, ix] = 1                  # 점이 하나라도 있으면 1
    return occ, z_edges

occ, ze = pixor_occupancy(P)
S = occ.shape[0]
fig, axes = plt.subplots(2, 4, figsize=(24, 11))
for j in range(S):
    s_ = S - 1 - j                                                    # 높은 구간부터
    a_ = axes.flat[j]
    a_.imshow(occ[s_], origin="lower", extent=[XMIN, XMAX, YMIN, YMAX], cmap="gray")
    a_.set_title(f"z in [{ze[s_]:.1f}, {ze[s_ + 1]:.1f}) m : {int(occ[s_].sum()):,} occupied cells")
axes.flat[S].imshow(occ.max(0), origin="lower", extent=[XMIN, XMAX, YMIN, YMAX], cmap="gray")
axes.flat[S].set_title("any height (all slices merged)")
for a_ in axes.flat[: S + 1]:
    draw_bev_boxes(a_, kitti["boxes"], color="red", lw=1); a_.set_xlim(XMIN, min(XMAX, P[:, 0].max())); a_.set_ylim(YMIN, YMAX)
plt.suptitle("PIXOR input: one binary BEV image per height slice (red = GT cars)", y=1.0)
plt.tight_layout(); plt.show()
print(f"PIXOR 입력 텐서: {S} 개 점유 구간 + 반사강도 1장 → ({S + 1}, {occ.shape[1]}, {occ.shape[2]}) — 이 '이미지' 를 2D CNN 에 바로 넣습니다")

> 🤔 **생각해보기** — PIXOR 점유 격자에서 차량(빨간 박스)은 주로 어느 높이 구간에 보이나요? 지면은 어느 구간에 보이나요?

**답변**

빨간 차량 박스 내부의 흰 점유 셀은 주로 z∈[-1.5, -1.0), [-1.0, -0.5), [-0.5, 0.0) m 구간에서 보인다. 차량의 표면이 여러 높이에 걸쳐 있으므로 한 구간에만 나타나는 것은 아니다. 지면은 주로 [-2.0, -1.5) m 구간에서 넓은 띠 모양으로 나타난다. 센서가 지면보다 약 1.73 m 높은 곳에 있으므로 지면의 z가 대략 -1.73 m인 것과도 맞는다. 빨간 박스 윤곽은 모든 높이 그림에 겹쳐 그려져 있으므로, 높이별 존재 여부는 박스 내부의 실제 흰 점유 셀을 기준으로 판단했다.


### PointRCNN : 원본 점에서 바로 — 1단계의 정답은 "점마다 전경/배경"
GT 박스 안에 있는 점 = 전경(foreground). PointRCNN 1단계는 모든 점을 전경/배경으로 나누고, **전경 점 하나하나가 자신이 속한 물체의 3D 박스를 제안**합니다.

In [ ]:
fgm = points_in_boxes(P, kitti["boxes"]); fg = fgm.any(1).numpy(); owner = fgm.float().argmax(1).numpy()
print(f"전경 점 {fg.sum():,} / {len(P):,} ({fg.mean():.1%}) — PointRCNN 1단계는 점마다 이 전경/배경을 맞히도록 학습합니다")
fig, ax = plt.subplots(figsize=(13, 9))
bev_ax(ax, None, "PointRCNN stage 1: foreground points (colored) and the box each one points to (arrows)", full=False)
ax.scatter(P[~fg, 0], P[~fg, 1], s=0.3, color="gray")
ax.scatter(P[fg, 0], P[fg, 1], s=1.5, c=[plt.cm.tab10(o % 10) for o in owner[fg]])
rng_ = np.random.default_rng(0); sel = rng_.choice(np.where(fg)[0], size=min(60, fg.sum()), replace=False)
for i_ in sel:
    bx = kitti["boxes"][owner[i_]]
    ax.annotate("", xy=(bx[0], bx[1]), xytext=(P[i_, 0], P[i_, 1]), arrowprops=dict(arrowstyle="->", color="white", lw=0.6, alpha=0.7))
draw_bev_boxes(ax, kitti["boxes"], color="white", lw=1)
ax.set_xlim(-2, min(XMAX, P[:, 0].max())); ax.set_ylim(P[:, 1].min(), P[:, 1].max()); plt.show()

## A4. LiDAR 시뮬레이터 — Part B 학습 데이터
레이저 한 줄 = 센서에서 출발하는 반직선(ray). 각 ray 가 **지면 / 차량 / 보행자 / 건물·기둥·덤불·트럭(라벨 없는 방해물)** 중 가장 먼저 닿는 곳에 점이 찍힙니다.
- 64 채널 (+2° ~ −24.9°), 방위각 0.25° 간격, 센서 높이 1.73 m — KITTI HDL-64E 와 동일한 기하
- 거리 측정 노이즈 2 cm, 무작위 점 손실 5%
- 내부 점이 5개 미만인 물체는 라벨에서 제외 (센서가 사실상 못 보는 물체)

In [ ]:
ELEV = np.radians(np.linspace(2.0, -24.9, 64)); AZIM = np.radians(np.arange(-90, 90, 0.25))
SENSOR_H = 1.73
_el, _az = np.meshgrid(ELEV, AZIM, indexing="ij")
RAYS = np.stack([np.cos(_el) * np.cos(_az), np.cos(_el) * np.sin(_az), np.sin(_el)], -1).reshape(-1, 3)

def raycast(boxes, refl, rng, max_range=80.0):
    d = RAYS; t = np.full(len(d), np.inf); I = np.zeros(len(d))
    g = d[:, 2] < 0; t[g] = -SENSOR_H / d[g, 2]; I[g] = rng.uniform(0.0, 0.15, g.sum())     # 지면
    if len(boxes):
        c, s = np.cos(boxes[:, 6]), np.sin(boxes[:, 6])
        dl = np.stack([d[:, :1] * c + d[:, 1:2] * s, -d[:, :1] * s + d[:, 1:2] * c, np.repeat(d[:, 2:3], len(boxes), 1)], -1)
        o = np.stack([-(c * boxes[:, 0] + s * boxes[:, 1]), -(-s * boxes[:, 0] + c * boxes[:, 1]), -boxes[:, 2]], -1)
        half = boxes[:, 3:6] / 2
        with np.errstate(divide="ignore", invalid="ignore"):                               # slab test (ray vs 회전 박스)
            t1, t2 = (-half - o) / dl, (half - o) / dl
            tmin = np.nanmax(np.minimum(t1, t2), -1); tmax = np.nanmin(np.maximum(t1, t2), -1)
        th = np.where((tmax >= tmin) & (tmin > 0), tmin, np.inf)
        best = th.argmin(1); tb = th[np.arange(len(d)), best]
        closer = tb < t; t[closer] = tb[closer]; I[closer] = refl[best[closer]]
    keep = (t < max_range) & (rng.random(len(d)) > 0.05)
    t = t[keep] + rng.normal(0, 0.02, keep.sum())
    return np.c_[d[keep] * t[:, None], np.clip(I[keep] + rng.normal(0, 0.03, keep.sum()), 0, 1)].astype(np.float32)

def sample_scene(seed):
    rng = np.random.default_rng(seed)
    objs = []                                                      # (class or None, box, reflectance)
    def free(x, y, r): return all(np.hypot(x - b[0], y - b[1]) > r + max(b[3], b[4]) / 2 + 0.3 for _, b, _ in objs)
    def add(cls, x, y, l, w, h, yaw, refl): objs.append((cls, np.array([x, y, -SENSOR_H + h / 2, l, w, h, yaw]), refl))
    for _ in range(rng.integers(4, 14) * 6):                       # 차량: 차선 위 or 길가 주차
        if sum(o[0] == "Car" for o in objs) >= 13: break
        x = rng.uniform(3, 58); y = rng.uniform(-8, 8) if rng.random() < 0.6 else rng.choice([-1, 1]) * rng.uniform(8.5, 11)
        yaw = rng.uniform(-np.pi, np.pi) if rng.random() < 0.25 else rng.choice([0, np.pi]) + rng.normal(0, 0.08)
        l, w, h = rng.normal(3.9, 0.3), rng.normal(1.62, 0.08), rng.normal(1.53, 0.08)
        if free(x, y, l / 2): add("Car", x, y, l, w, h, yaw, rng.uniform(0.2, 0.7))
    for _ in range(rng.integers(0, 8)):                            # 보행자: 인도 or 횡단
        x = rng.uniform(3, 45); y = rng.choice([-1, 1]) * rng.uniform(11, 14) if rng.random() < 0.7 else rng.uniform(-8, 8)
        if free(x, y, 0.5): add("Pedestrian", x, y, rng.normal(0.8, 0.08), rng.normal(0.6, 0.06), rng.normal(1.73, 0.07),
                                rng.uniform(-np.pi, np.pi), rng.uniform(0.1, 0.5))
    for _ in range(rng.integers(0, 3)):                            # 트럭 (라벨 없음 = 배경)
        x, y = rng.uniform(8, 55), rng.uniform(-7, 7)
        if free(x, y, 4): add(None, x, y, rng.uniform(6, 10), 2.4, rng.uniform(2.6, 3.5), rng.choice([0, np.pi]), 0.5)
    for side in [-1, 1]:                                           # 건물 벽
        x = rng.uniform(-5, 5)
        while x < 60:
            L = rng.uniform(5, 20)
            if rng.random() < 0.8: add(None, x + L / 2, side * rng.uniform(15, 22), L, rng.uniform(0.5, 1.5), rng.uniform(3, 10), rng.normal(0, 0.05), rng.uniform(0.1, 0.6))
            x += L + rng.uniform(1, 8)
    for _ in range(rng.integers(5, 15)):                           # 기둥 / 덤불
        x, y = rng.uniform(2, 58), rng.choice([-1, 1]) * rng.uniform(10.5, 16)
        if rng.random() < 0.5: add(None, x, y, 0.3, 0.3, rng.uniform(3, 6), 0.0, rng.uniform(0.3, 0.8))
        elif free(x, y, 1.5): add(None, x, y, rng.uniform(0.8, 3), rng.uniform(0.8, 2), rng.uniform(0.5, 1.5), rng.uniform(-np.pi, np.pi), rng.uniform(0.05, 0.3))
    allb = np.array([b for _, b, _ in objs]).reshape(-1, 7)
    pts = raycast(allb, np.array([r for _, _, r in objs]), rng)
    lab = [(CLASSES.index(c), b) for c, b, _ in objs if c is not None]
    gt = np.array([b for _, b in lab], dtype=np.float32).reshape(-1, 7); gl = np.array([c for c, _ in lab], dtype=np.int64)
    npts = points_in_boxes(pts, gt).sum(0).numpy() if len(gt) else np.zeros(0)
    keep = npts >= 5
    clutter = np.array([b for c, b, _ in objs if c is None], dtype=np.float32).reshape(-1, 7)
    return dict(points=pts, boxes=gt[keep], labels=gl[keep], n_pts=npts[keep], clutter=clutter,
                dropped=gt[~keep], dropped_labels=gl[~keep])

t0 = time.time(); demo = sample_scene(12345)
print(f"한 장면 생성 {time.time() - t0:.2f}s | points {len(demo['points']):,} | Car {int((demo['labels'] == 0).sum())}, "
      f"Ped {int((demo['labels'] == 1).sum())} | 점이 5개 미만이라 라벨에서 제외된 물체 {len(demo['dropped'])}")
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
bev_ax(axes[0], demo["points"], "Synthetic scene: points (color = height)", s=0.6)
bev_ax(axes[1], demo["points"], "with labels (green=Car, cyan=Ped, gray=unlabeled clutter, red=too few points)", s=0.6)
draw_bev_boxes(axes[1], demo["clutter"], color="gray", lw=1)
draw_bev_boxes(axes[1], demo["boxes"], color=["lime" if l == 0 else "cyan" for l in demo["labels"]], lw=1.8)
draw_bev_boxes(axes[1], demo["dropped"], color="red", lw=1.5, ls="--")
for ax in axes: ax.plot(0, 0, "r^", ms=10)
plt.tight_layout(); plt.show()
plot3d(demo["points"], [(demo["boxes"], "lime", "GT"), (demo["clutter"], "gray", "clutter")], title="Synthetic scene (3D)")

In [ ]:
print(f"합성 데이터 생성 중: train {N_TRAIN} + val {N_VAL} 장면 ...")
t0 = time.time()
train_set = [sample_scene(1000 + i) for i in range(N_TRAIN)]
val_set   = [sample_scene(900000 + i) for i in range(N_VAL)]
print(f"완료 {time.time() - t0:.0f}s | 평균 점 수 {np.mean([len(s['points']) for s in train_set]):,.0f} "
      f"(실제 KITTI 프레임: {len(P):,})")

# Sparsity: 거리 vs 물체 내부 점 수
dist = np.concatenate([np.hypot(s["boxes"][:, 0], s["boxes"][:, 1]) for s in train_set])
npt = np.concatenate([s["n_pts"] for s in train_set]); lab = np.concatenate([s["labels"] for s in train_set])
fig, ax = plt.subplots(figsize=(9, 4.5))
for c, col in [(0, "tab:green"), (1, "tab:cyan")]:
    ax.scatter(dist[lab == c], npt[lab == c], s=4, alpha=0.4, color=col, label=f"synthetic {CLASSES[c]}")
kd = np.hypot(kitti["boxes"][:, 0], kitti["boxes"][:, 1])
ax.scatter(kd, points_in_boxes(P, kitti["boxes"]).sum(0).numpy(), s=60, marker="*", color="red", label="real KITTI frame")
ax.set_yscale("log"); ax.set_xlabel("distance from LiDAR (m)"); ax.set_ylabel("# points inside the box (log)")
ax.legend(); ax.set_title("Sparsity: far objects get very few points"); plt.show()

> 🤔 **생각해보기** — 거리가 멀어질수록 물체 안의 점 수는 어떻게 변하나요? 30 m 보다 멀리 있는 차량은 대략 몇 개의 점으로 표현되나요?
> 실제 KITTI 프레임(빨간 별)도 합성 데이터와 비슷한 경향을 보이나요?

**답변**

거리가 멀어질수록 물체 내부의 점 수가 전반적으로 줄어든다. LiDAR의 각도 간격이 일정하면 멀리 있는 물체가 차지하는 각도 범위가 작아져, 표면에 닿는 레이저 수가 감소하기 때문이다. 합성 차량의 산점도를 보면 30 m보다 먼 차량은 대체로 수십 개의 점으로 표현되며, 약 10~100개 수준에 많이 분포한다. 같은 거리에서도 가림과 물체 방향 등에 따라 차이가 있고, 5개 안팎까지 줄어든 차량도 보인다. 실제 KITTI의 빨간 별도 전체적으로 비슷한 감소 경향을 보인다. A1의 실행 출력에서 확인된 값을 보면, 8.2 m 차량은 1,933개, 21.9 m 차량은 169개, 34.3 m 차량은 54개의 내부 점을 갖는다. 다만 실제 데이터는 한 프레임의 관찰이며, 거리만으로 모든 점 수 차이를 설명할 수는 없다. 합성 데이터는 점이 5개 미만인 물체를 라벨에서 제외하므로, 그래프의 하한이 5개라는 점도 고려해야 한다.


# Part B. VoxelNet 을 처음부터 끝까지

```
point cloud (N×4)
 ─▶ ① voxel partition & grouping  ─▶ (V voxels × 32 points × 7)
 ─▶ ② VFE-1 → VFE-2 → FCN + max   ─▶ (V × 64)                  ← 점 집합에서 feature 를 "학습"
 ─▶ ③ sparse → dense 4D tensor     ─▶ (64 × 10 × 256 × 256)
 ─▶ ④ 3D conv middle layers        ─▶ (32 × 3 × 256 × 256) → reshape (96 × 256 × 256)   ← 높이 방향 문맥을 모아 BEV 로
 ─▶ ⑤ RPN (2D CNN)  ─▶ ⑥ anchor 별 score + 7-dof box  ─▶ NMS
```
⑤부터는 **실습 1과 같은 2D detection** 입니다.
> 논문 설정(voxel 0.2×0.2×0.4 m, VFE 2층 + FCN, 3D conv 3층, RPN 3블록)에서 Colab 에 맞게 채널 수와 범위만 줄였습니다.
> 논문은 Car 와 Pedestrian/Cyclist 를 별도 네트워크로 학습하지만, 여기서는 한 네트워크에서 함께 학습합니다.

## B1. Voxel partition & grouping
공간을 0.2 × 0.2 × 0.4 m voxel 로 나누고 점을 자신이 속한 voxel 로 묶습니다. 점이 T(=32)개보다 많은 voxel 은 **무작위로 T개만** 남깁니다
(논문의 random sampling — 계산량을 줄이고, 가까운 물체에 점이 몰리는 편향을 줄임).

In [ ]:
def voxelize(points, shuffle=True):
    # points: (N, 4) tensor → voxels (V, MAX_PTS, 4), npts (V,), coords (V, 3) = (iz, iy, ix)
    p = points
    m = (p[:, 0] >= XMIN) & (p[:, 0] < XMAX) & (p[:, 1] >= YMIN) & (p[:, 1] < YMAX) & (p[:, 2] >= ZMIN) & (p[:, 2] < ZMAX)
    p = p[m]
    if shuffle: p = p[torch.randperm(len(p), device=p.device)]              # random sampling 을 위해 섞음
    ix = ((p[:, 0] - XMIN) / VX).long().clamp(0, NX - 1); iy = ((p[:, 1] - YMIN) / VY).long().clamp(0, NY - 1)
    iz = ((p[:, 2] - ZMIN) / VZ).long().clamp(0, NZ - 1)
    vid = (iz * NY + iy) * NX + ix
    uniq, inv, counts = torch.unique(vid, return_inverse=True, return_counts=True)
    order = torch.argsort(inv, stable=True); inv_s = inv[order]
    starts = torch.cumsum(counts, 0) - counts
    rank = torch.arange(len(p), device=p.device) - starts[inv_s]             # voxel 안에서 몇 번째 점인가
    keep = rank < MAX_PTS
    vox = torch.zeros(len(uniq), MAX_PTS, 4, device=p.device)
    vox[inv_s[keep], rank[keep]] = p[order][keep]
    npts = counts.clamp(max=MAX_PTS)
    coords = torch.stack([uniq // (NY * NX), (uniq // NX) % NY, uniq % NX], 1)
    if len(uniq) > MAX_VOXELS:
        sel = torch.randperm(len(uniq), device=p.device)[:MAX_VOXELS]
        vox, npts, coords, counts = vox[sel], npts[sel], coords[sel], counts[sel]
    return vox, npts, coords, dict(n_in=len(points), n_range=len(p), counts=counts)

def voxel_features(vox, npts):
    # 논문의 7차원 입력: (x, y, z, r, x - v_x, y - v_y, z - v_z),  v = voxel 안 점들의 중심(centroid)
    mask = (torch.arange(MAX_PTS, device=vox.device)[None] < npts[:, None]).float()
    xyz = vox[..., :3]
    mean = (xyz * mask[..., None]).sum(1) / npts[:, None].float()
    r = vox[..., 3:4] if USE_INTENSITY else torch.zeros_like(vox[..., 3:4])
    return torch.cat([xyz, r, xyz - mean[:, None]], -1) * mask[..., None], mask

scene = val_set[0]
pts_t = torch.from_numpy(scene["points"]).to(device)
vox, npts, coords, st = voxelize(pts_t)
print(f"입력 점 {st['n_in']:,} → 범위 안 {st['n_range']:,} → 비어있지 않은 voxel {len(npts):,} / {NX * NY * NZ:,} "
      f"({len(npts) / (NX * NY * NZ):.2%})")
print(f"voxel 당 점 수: 평균 {st['counts'].float().mean():.1f}, 최대 {int(st['counts'].max())}, "
      f"T={MAX_PTS} 초과로 샘플링된 voxel {int((st['counts'] > MAX_PTS).sum())}")
print("voxel tensor:", tuple(vox.shape), "| coords (iz, iy, ix):", tuple(coords.shape))

occ3 = torch.zeros(NZ, NY, NX); occ3[coords[:, 0].cpu(), coords[:, 1].cpu(), coords[:, 2].cpu()] = npts.float().cpu()
fig, axes = plt.subplots(2, (NZ + 1) // 2, figsize=(5 * ((NZ + 1) // 2), 10))
for z in range(NZ):
    a_ = axes.flat[NZ - 1 - z]
    a_.imshow(occ3[z].clamp(max=MAX_PTS), origin="lower", extent=[XMIN, XMAX, YMIN, YMAX], cmap="magma")
    a_.set_title(f"z in [{ZMIN + z * VZ:.1f}, {ZMIN + (z + 1) * VZ:.1f}) m : {int((occ3[z] > 0).sum()):,} voxels", fontsize=10)
plt.suptitle("Non-empty voxels per height slice (top-left = highest). Ground shows up only in the low slices.", y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
b = torch.as_tensor(scene["boxes"][scene["labels"] == 0][0])
fig, axes = plt.subplots(1, 3, figsize=(24, 7), gridspec_kw=dict(width_ratios=[0.8, 1, 1]))
axes[0].hist(st["counts"].cpu().numpy(), bins=np.arange(1, 80), color="tab:orange", log=True)
axes[0].axvline(MAX_PTS, color="r", ls="--", label=f"T={MAX_PTS}"); axes[0].legend()
axes[0].set_xlabel("# points in a voxel"); axes[0].set_ylabel("# voxels (log)"); axes[0].set_title("Points per non-empty voxel")
bev_ax(axes[1], scene["points"], "Zoom on one car (top view): 0.2 m voxel grid", s=6, full=False)
draw_bev_boxes(axes[1], b, color="lime", lw=2)
x0_, y0_ = XMIN + np.floor((float(b[0]) - 3 - XMIN) / VX) * VX, YMIN + np.floor((float(b[1]) - 3 - YMIN) / VY) * VY
axes[1].set_xticks(x0_ + np.arange(31) * VX, minor=True); axes[1].set_yticks(y0_ + np.arange(31) * VY, minor=True)
axes[1].grid(which="minor", color="w", alpha=0.25, lw=0.5)
axes[1].set_xlim(float(b[0]) - 3, float(b[0]) + 3); axes[1].set_ylim(float(b[1]) - 3, float(b[1]) + 3)
pn = scene["points"]; near = (np.abs(pn[:, 0] - float(b[0])) < 3) & (np.abs(pn[:, 1] - float(b[1])) < 3)
axes[2].scatter(pn[near, 0], pn[near, 2], s=6, c=pn[near, 2], cmap="viridis", vmin=-2.2, vmax=1.0)
cx_ = box_corners_bev(b)[0, :, 0]
axes[2].add_patch(patches.Rectangle((float(cx_.min()), float(b[2] - b[5] / 2)), float(cx_.max() - cx_.min()), float(b[5]),
                                    fill=False, edgecolor="lime", lw=2))
for k_ in range(NZ + 1): axes[2].axhline(ZMIN + k_ * VZ, color="w", alpha=0.3, lw=0.6)
axes[2].set_xticks(x0_ + np.arange(31) * VX, minor=True); axes[2].grid(which="minor", axis="x", color="w", alpha=0.2, lw=0.5)
axes[2].set_xlim(float(b[0]) - 3, float(b[0]) + 3); axes[2].set_ylim(ZMIN, ZMAX); axes[2].set_aspect("equal")
axes[2].set_facecolor("#111111"); axes[2].set_xlabel("x (m)"); axes[2].set_ylabel("z (m)")
axes[2].set_title(f"Side view: {VZ} m height bins → {NZ} voxel layers")
plt.tight_layout(); plt.show()

## B2. VFE 의 입력 — voxel 하나 안의 점들
점마다 7차원: 자기 좌표 (x, y, z), 반사강도 r, 그리고 **voxel 안 점들의 중심(centroid)으로부터의 오프셋**.
점이 T 개보다 적은 voxel 은 0 으로 채우고(padding), mask 로 구분합니다.

In [ ]:
feat7, mask = voxel_features(vox, npts)
car0 = torch.as_tensor(scene["boxes"][scene["labels"] == 0][0]).to(device)
vc = torch.stack([coords[:, 2].float() * VX + VX / 2 + XMIN, coords[:, 1].float() * VY + VY / 2 + YMIN,
                  coords[:, 0].float() * VZ + VZ / 2 + ZMIN], 1)
cand_ = torch.where(npts >= 4)[0]
k = int(cand_[(vc[cand_] - car0[:3]).norm(dim=1).argmin()])              # 첫 번째 차량 위의 voxel
names7 = ["x", "y", "z", "r", "x-v_x", "y-v_y", "z-v_z"]
print(f"voxel #{k} (iz={int(coords[k, 0])}, iy={int(coords[k, 1])}, ix={int(coords[k, 2])}), 점 {int(npts[k])}개의 7차원 입력:")
print("      " + "".join(f"{n:>8s}" for n in names7))
for j in range(min(int(npts[k]) + 1, 8)):
    print(f"  p{j:<3d}" + "".join(f"{v:8.2f}" for v in feat7[k, j].tolist()) + ("   ← padding (0)" if j >= npts[k] else ""))

## B3. 모델 정의 — VFE · sparse→dense · 3D conv middle layers · RPN

**VFE layer** (논문 Fig. 3): 점마다 FCN(Linear + BN + ReLU) → **point-wise feature** →
voxel 안에서 element-wise max → **locally aggregated feature** → 두 feature 를 점마다 이어붙임.
이렇게 하면 각 점이 "자기 자신" 과 "같은 voxel 의 다른 점들 전체" 를 함께 보게 됩니다. VFE 를 두 번 쌓고, 마지막 FCN + max 로 voxel 당 벡터 하나를 만듭니다.

In [ ]:
class VFELayer(nn.Module):
    def __init__(self, cin, cout):
        super().__init__(); self.lin = nn.Linear(cin, cout // 2, bias=False); self.bn = nn.BatchNorm1d(cout // 2, eps=1e-3, momentum=0.01)
    def forward(self, x, mask):
        pw = self.lin(x)
        pw = F.relu(self.bn(pw.view(-1, pw.shape[-1])).view(pw.shape)) * mask[..., None]     # point-wise feature
        agg = pw.max(1, keepdim=True).values                                                  # locally aggregated feature
        return torch.cat([pw, agg.expand_as(pw)], -1) * mask[..., None]

class VFE(nn.Module):
    def __init__(self):
        super().__init__()
        self.vfe1, self.vfe2 = VFELayer(7, 32), VFELayer(32, 64)
        self.fcn = nn.Linear(64, 64, bias=False); self.bn = nn.BatchNorm1d(64, eps=1e-3, momentum=0.01)
    def forward(self, x, mask):
        h1 = self.vfe1(x, mask); h2 = self.vfe2(h1, mask)
        h = self.fcn(h2); h = F.relu(self.bn(h.view(-1, 64)).view(h.shape)) * mask[..., None]
        v, arg = h.max(1)
        return v, arg, h1

def conv3d(cin, cout, stride):
    return nn.Sequential(nn.Conv3d(cin, cout, 3, stride, 1, bias=False), nn.BatchNorm3d(cout, eps=1e-3, momentum=0.01), nn.ReLU())

def conv_block(cin, cout, n, stride):
    layers = [nn.Conv2d(cin, cout, 3, stride, 1, bias=False), nn.BatchNorm2d(cout, eps=1e-3, momentum=0.01), nn.ReLU()]
    for _ in range(n):
        layers += [nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout, eps=1e-3, momentum=0.01), nn.ReLU()]
    return nn.Sequential(*layers)

def up_block(cin, cout, s):
    return nn.Sequential(nn.ConvTranspose2d(cin, cout, s, s, bias=False), nn.BatchNorm2d(cout, eps=1e-3, momentum=0.01), nn.ReLU())

D_OUT = ((NZ - 1) // 2 + 1 - 1) // 2 + 1                  # 3D conv 2번의 z-stride 2 를 거친 뒤 높이 칸 수 (10 → 5 → 3)

# ---- 3D anchors: RPN 출력 격자(128x128)의 각 칸마다 클래스별 2개(0°, 90°) = 4개 ----
FH, FW = NY // 2, NX // 2
def make_anchors():
    ys = YMIN + (torch.arange(FH) + 0.5) * (YMAX - YMIN) / FH; xs = XMIN + (torch.arange(FW) + 0.5) * (XMAX - XMIN) / FW
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    al, ac = [], []
    for ci, c in enumerate(CLASSES):
        for rot in [0.0, np.pi / 2]:
            l, w, h = ANCHOR_SIZE[c]
            al.append(torch.stack([gx, gy, torch.full_like(gx, ANCHOR_Z[c]), torch.full_like(gx, l),
                                   torch.full_like(gx, w), torch.full_like(gx, h), torch.full_like(gx, rot)], -1))
            ac.append(ci)
    return torch.stack(al, 2).reshape(-1, 7).to(device), torch.tensor(ac).repeat(FH * FW).to(device)
ANCHORS, ANCHOR_CLS = make_anchors()
NA = 2 * len(CLASSES)

class VoxelNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.vfe = VFE()
        self.mid = nn.ModuleList([conv3d(64, 32, (2, 1, 1)), conv3d(32, 32, (1, 1, 1)), conv3d(32, 32, (2, 1, 1))])
        self.b1, self.b2, self.b3 = conv_block(32 * D_OUT, 64, 3, 2), conv_block(64, 128, 5, 2), conv_block(128, 256, 5, 2)
        self.u1, self.u2, self.u3 = up_block(64, 128, 1), up_block(128, 128, 2), up_block(256, 128, 4)
        self.cls = nn.Conv2d(384, NA, 1); self.box = nn.Conv2d(384, NA * 7, 1)
        nn.init.normal_(self.box.weight, std=0.001); nn.init.zeros_(self.box.bias)
        if CLS_LOSS == "focal": nn.init.constant_(self.cls.bias, -math.log((1 - 0.01) / 0.01))   # 실습 1 의 RetinaNet prior 초기화

    def forward(self, batch_points, return_all=False):
        B = len(batch_points); fs, ms, cs, extra = [], [], [], []
        for b, pts in enumerate(batch_points):
            vx, n, co, st = voxelize(pts)
            f, m = voxel_features(vx, n)
            fs.append(f); ms.append(m); cs.append(torch.cat([torch.full_like(co[:, :1], b), co], 1)); extra.append(st)
        f, m, co = torch.cat(fs), torch.cat(ms), torch.cat(cs)
        vf, arg, h1 = self.vfe(f, m)                                           # (V, 64)
        dense = torch.zeros(B * NZ * NY * NX, vf.shape[1], device=vf.device, dtype=vf.dtype)
        dense[((co[:, 0] * NZ + co[:, 1]) * NY + co[:, 2]) * NX + co[:, 3]] = vf   # sparse → dense
        dense = dense.view(B, NZ, NY, NX, -1).permute(0, 4, 1, 2, 3)          # (B, C, D, H, W)
        m1 = self.mid[0](dense); m2 = self.mid[1](m1); m3 = self.mid[2](m2)
        bev = m3.flatten(1, 2)                                                 # (B, 32 x D_OUT, H, W)
        x1 = self.b1(bev); x2 = self.b2(x1); x3 = self.b3(x2)
        x = torch.cat([self.u1(x1), self.u2(x2), self.u3(x3)], 1)
        def flat(t, k): return t.permute(0, 2, 3, 1).reshape(B, -1, k)
        out = dict(cls=flat(self.cls(x), 1)[..., 0], box=flat(self.box(x), 7))
        if return_all:
            out.update(dense=dense, m1=m1, m2=m2, m3=m3, bev=bev, x1=x1, x2=x2, x3=x3, neck=x,
                       voxel_feat=vf, vfe_arg=arg, vfe1=h1, coords=co, stats=extra)
        return out

def n_params(m): return sum(p.numel() for p in m.parameters()) / 1e6
seed_all(); model_init = VoxelNet().to(device).eval()
INIT_STATE = {k_: v.detach().cpu().clone() for k_, v in model_init.state_dict().items()}
print(f"VoxelNet params: {n_params(model_init):.2f}M  (VFE {n_params(model_init.vfe) * 1e3:.1f}K, 3D conv {n_params(model_init.mid) * 1e3:.0f}K)")
print(f"anchors: {FH} x {FW} cells x {NA} = {len(ANCHORS):,}  (per location: {[CLASSES[i] for i in ANCHOR_CLS[:NA].tolist()]})")

In [ ]:
with torch.no_grad():
    O = model_init([pts_t], return_all=True)
print("voxel features    :", tuple(O["voxel_feat"].shape), "  ← (V voxels, 64)")
print("dense 4D tensor   :", tuple(O["dense"].shape), "  ← (B, C, D, H, W)")
for k_, t_ in [("m1", "3D conv 1 (z-stride 2)"), ("m2", "3D conv 2"), ("m3", "3D conv 3 (z-stride 2)")]:
    print(f"{t_:<18s}:", tuple(O[k_].shape))
print("BEV after reshape :", tuple(O["bev"].shape), "  ← (C x D) 를 채널로 합침 — 이제부터는 2D CNN (RPN)")
for k_ in ["x1", "x2", "x3", "neck"]: print(f"{k_:<18s}:", tuple(O[k_].shape))
print("head outputs      :", {k_: tuple(O[k_].shape) for k_ in ["cls", "box"]})

# VFE-1 의 출력: 앞 16채널은 점마다 다르고(point-wise), 뒤 16채널은 모든 점이 같다(aggregated)
h1 = O["vfe1"][k][: int(npts[k]) + 2].cpu()
fig, axes = plt.subplots(1, 2, figsize=(18, 4), gridspec_kw=dict(width_ratios=[1, 1.4]))
axes[0].imshow(h1, cmap="viridis", aspect="auto"); axes[0].axvline(15.5, color="r", lw=2)
axes[0].set_xlabel("VFE-1 output channel (0-15 point-wise | 16-31 locally aggregated)"); axes[0].set_ylabel("point in voxel (last rows = padding)")
axes[0].set_title(f"VFE-1 output for voxel #{k} ({int(npts[k])} points)")
win = torch.bincount(O["vfe_arg"][k].cpu(), minlength=MAX_PTS)[: int(npts[k])]
axes[1].bar(range(len(win)), win.numpy(), color="tab:orange"); axes[1].set_xlabel("point index in voxel")
axes[1].set_ylabel("# of the 64 output channels won (max)"); axes[1].set_title("Final max-pooling: which point dominates each channel")
plt.tight_layout(); plt.show()

ext = [XMIN, XMAX, YMIN, YMAX]
maps = [(O["dense"][0].abs().mean(0).max(0).values, "dense voxel features (max over z)")] + \
       [(O["m3"][0, :, d].abs().mean(0), f"after 3D convs: height layer {d}") for d in range(O["m3"].shape[2])] + \
       [(O[k_][0].abs().mean(0), t_) for k_, t_ in [("x1", "RPN block1 (128x128)"), ("x2", "RPN block2 (64x64)"),
                                                     ("x3", "RPN block3 (32x32)"), ("neck", "RPN concat (128x128)")]]
fig, axes = plt.subplots(2, 4, figsize=(24, 11))
for a_, (im_, t_) in zip(axes.flat, maps):
    a_.imshow(im_.cpu(), origin="lower", extent=ext, cmap="viridis"); a_.set_title(t_); a_.set_xlabel("x (m)")
for a_ in list(axes.flat)[len(maps):]: a_.axis("off")
plt.suptitle("Untrained model: feature maps already show the scene geometry (they start as a scattered copy of the points)", y=1.0)
plt.tight_layout(); plt.show()

## B4. 3D anchor 와 학습 target
실습 1의 anchor 와 원리는 같고, 차원만 늘었습니다.

| | 2D (실습 1) | 3D VoxelNet |
|---|---|---|
| anchor | (x, y, w, h) — 크기 × 비율 | (x, y, z, l, w, h, **yaw**) — **클래스별 실제 크기 1개** × 회전 2개 (0°, 90°) |
| 매칭 | 이미지 IoU | **BEV IoU** (Car: ≥0.6 positive / <0.45 negative, Pedestrian: 0.5 / 0.35) |
| regression | 4개 | 7개: $\Delta x = \frac{x_g - x_a}{d_a}, \Delta y = \frac{y_g - y_a}{d_a}, \Delta z = \frac{z_g - z_a}{h_a}$ ($d_a = \sqrt{l_a^2 + w_a^2}$), $\log\frac{l_g}{l_a}$, $\log\frac{w_g}{w_a}$, $\log\frac{h_g}{h_a}$, $\Delta\theta = \theta_g - \theta_a$ |

> 매칭 IoU 는 회전 박스를 가까운 축(0° 또는 90°)으로 돌린 축정렬 BEV 박스로 근사합니다.
> $\Delta\theta$ 는 π 주기로 정규화합니다 — 즉 VoxelNet 은 차량의 **앞과 뒤를 구분하지 않습니다.**

In [ ]:
def bev_aabb(b):
    swap = torch.abs(torch.sin(b[:, 6])) > torch.abs(torch.cos(b[:, 6]))
    l = torch.where(swap, b[:, 4], b[:, 3]); w = torch.where(swap, b[:, 3], b[:, 4])
    return torch.stack([b[:, 0] - l / 2, b[:, 1] - w / 2, b[:, 0] + l / 2, b[:, 1] + w / 2], 1)

def limit_period(v, offset=0.5, period=np.pi): return v - torch.floor(v / period + offset) * period

def encode(gt, a):
    da = torch.sqrt(a[:, 3] ** 2 + a[:, 4] ** 2)
    return torch.stack([(gt[:, 0] - a[:, 0]) / da, (gt[:, 1] - a[:, 1]) / da, (gt[:, 2] - a[:, 2]) / a[:, 5],
                        torch.log(gt[:, 3] / a[:, 3]), torch.log(gt[:, 4] / a[:, 4]), torch.log(gt[:, 5] / a[:, 5]),
                        limit_period(gt[:, 6] - a[:, 6])], 1)

def decode(t, a):
    da = torch.sqrt(a[:, 3] ** 2 + a[:, 4] ** 2)
    return torch.stack([t[:, 0] * da + a[:, 0], t[:, 1] * da + a[:, 1], t[:, 2] * a[:, 5] + a[:, 2],
                        torch.exp(t[:, 3]) * a[:, 3], torch.exp(t[:, 4]) * a[:, 4], torch.exp(t[:, 5]) * a[:, 5],
                        t[:, 6] + a[:, 6]], 1)

def assign(gt_boxes, gt_labels):
    # 반환: label (1 pos / 0 neg / -1 ignore), 매칭된 GT index, 각 anchor 의 max IoU
    N = len(ANCHORS); label = torch.zeros(N, dtype=torch.long, device=device)
    midx = torch.full((N,), -1, dtype=torch.long, device=device); maxiou = torch.zeros(N, device=device)
    for ci, c in enumerate(CLASSES):
        am = torch.where(ANCHOR_CLS == ci)[0]; gm = torch.where(gt_labels == ci)[0]
        if len(gm) == 0: continue
        iou = box_iou(bev_aabb(ANCHORS[am]), bev_aabb(gt_boxes[gm]))
        mx, ag = iou.max(1); hi, lo = IOU_THR[c]
        pos = mx >= hi
        best = iou.max(0).values
        pos |= ((iou == best[None]) & (best[None] > 0)).any(1)               # 각 GT 의 최고 IoU anchor 는 positive
        label[am] = torch.where(pos, 1, torch.where(mx < lo, 0, -1)); midx[am] = torch.where(pos, gm[ag], -1); maxiou[am] = mx
    return label, midx, maxiou

gtb = torch.from_numpy(scene["boxes"]).to(device); gtl = torch.from_numpy(scene["labels"]).to(device)
lab, midx, maxiou = assign(gtb, gtl)
pos = lab == 1
print(f"anchors {len(lab):,}: positive {int(pos.sum())}, negative {int((lab == 0).sum()):,}, ignore {int((lab == -1).sum())}")
for ci, c in enumerate(CLASSES):
    print(f"  {c:<10s}: GT {int((gtl == ci).sum())}, positive anchors {int((pos & (ANCHOR_CLS == ci)).sum())}")
print("GT 별 positive anchor 수:", torch.bincount(midx[pos], minlength=len(gtb)).tolist())

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
bev_ax(axes[0], scene["points"], "4 anchors at one location (Car 0°/90°, Ped 0°/90°) + anchor centers (every 4th)", s=0.4)
sub = ANCHORS.view(FH, FW, NA, 7)[::4, ::4, 0].reshape(-1, 7).cpu()
axes[0].scatter(sub[:, 0], sub[:, 1], s=0.5, color="white", alpha=0.5)
g0 = gtb[gtl == 0][0]; iy = int((g0[1] - YMIN) / (YMAX - YMIN) * FH); ix = int((g0[0] - XMIN) / (XMAX - XMIN) * FW)
draw_bev_boxes(axes[0], ANCHORS.view(FH, FW, NA, 7)[iy, ix], color=["orange", "orange", "cyan", "cyan"], lw=1.5)
draw_bev_boxes(axes[0], g0, color="lime", lw=2.5)
bev_ax(axes[1], scene["points"], "Positive anchors (colored by matched GT) + GT (white)", s=0.4)
pi = torch.where(pos)[0]; cols = plt.cm.tab20(np.arange(len(gtb)) % 20)
draw_bev_boxes(axes[1], ANCHORS[pi], color=[cols[int(midx[i])] for i in pi], lw=0.8)
draw_bev_boxes(axes[1], gtb, color="white", lw=1.8)
plt.tight_layout(); plt.show()

i = int(pi[ANCHOR_CLS[pi] == 0][0]); a, g = ANCHORS[i:i+1], gtb[midx[i]:midx[i]+1]; t = encode(g, a)[0]
print(f"\nCar anchor {np.round(a[0].tolist(), 2)}\nGT         {np.round(g[0].tolist(), 2)}")
print("target (Δx, Δy, Δz, log l, log w, log h, Δθ) =", np.round(t.tolist(), 3))
rec = decode(t[None], a)[0]
print("decode 결과:", np.round(rec.tolist(), 2), "| yaw 는 GT 와 같거나 π 만큼 다름 (앞/뒤 구분 없음):",
      round(float(limit_period(rec[6] - g[0, 6])), 4))

## B5. Loss · 후처리
VoxelNet 의 손실:
$$L = \frac{\alpha}{N_{pos}} \sum_{i \in pos} L_{cls}(p_i, 1) + \frac{\beta}{N_{neg}} \sum_{j \in neg} L_{cls}(p_j, 0) + \frac{1}{N_{pos}} \sum_{i \in pos} L_{reg}(\mathbf{u}_i, \mathbf{u}_i^*)$$
positive 와 negative 의 binary cross entropy 를 **각각 평균**낸 뒤 가중합합니다 (α = 1.5, β = 1). 수만 개의 negative 가 수십 개의 positive 를 압도하지 못하게 하는 방법으로,
실습 1 의 RetinaNet focal loss 와는 다른 해법입니다. 회귀는 smooth-L1 입니다.

후처리: score threshold → (축정렬 BEV 근사) NMS

In [ ]:
def compute_loss(out, batch_gt):
    B = out["cls"].shape[0]; Lp = Ln = Lb = 0.0; npos = nneg = 0
    for b in range(B):
        gtb_, gtl_ = batch_gt[b]
        lab_, midx_, _ = assign(gtb_, gtl_)
        p_, n_ = lab_ == 1, lab_ == 0
        logit = out["cls"][b].float()
        if CLS_LOSS == "focal":
            v_ = p_ | n_
            Lp = Lp + sigmoid_focal_loss(logit[v_], p_[v_].float(), alpha=0.25, gamma=2.0, reduction="sum")
        else:
            Lp = Lp + F.binary_cross_entropy_with_logits(logit[p_], torch.ones_like(logit[p_]), reduction="sum")
            Ln = Ln + F.binary_cross_entropy_with_logits(logit[n_], torch.zeros_like(logit[n_]), reduction="sum")
        npos += int(p_.sum()); nneg += int(n_.sum())
        if p_.any():
            Lb = Lb + F.smooth_l1_loss(out["box"][b][p_].float(), encode(gtb_[midx_[p_]], ANCHORS[p_]), beta=1 / 9, reduction="sum")
    if CLS_LOSS == "focal":
        return dict(loss_cls=Lp / max(npos, 1), loss_box=Lb / max(npos, 1))
    return dict(loss_pos=LOSS_ALPHA * Lp / max(npos, 1), loss_neg=LOSS_BETA * Ln / max(nneg, 1), loss_box=Lb / max(npos, 1))

@torch.no_grad()
def postprocess(out, b=0, score_thr=0.1, nms_iou=0.2, topk=500, max_det=100):
    sc = torch.sigmoid(out["cls"][b].float()); keep = torch.where(sc > score_thr)[0]
    keep = keep[sc[keep].topk(min(topk, len(keep))).indices] if len(keep) else keep
    boxes = decode(out["box"][b][keep].float(), ANCHORS[keep]); cls = ANCHOR_CLS[keep]
    k2 = batched_nms(bev_aabb(boxes), sc[keep], cls, nms_iou)[:max_det]
    return dict(boxes=boxes[k2], scores=sc[keep][k2], labels=cls[k2])

with torch.no_grad():
    out0 = model_init([pts_t]); l0 = compute_loss(out0, [(gtb, gtl)])
print("학습 전 loss:", {k_: round(float(v), 3) for k_, v in l0.items()})
npos_, nneg_ = int(pos.sum()), int((lab == 0).sum())
print(f"positive {npos_} vs negative {nneg_:,}:")
print(f"  모든 anchor 를 한꺼번에 평균내면 positive 가 차지하는 loss 비중 ≈ {npos_ / (npos_ + nneg_):.2%}")
print(f"  VoxelNet 처럼 따로 평균내면 (α={LOSS_ALPHA}, β={LOSS_BETA}) positive 비중 ≈ {LOSS_ALPHA / (LOSS_ALPHA + LOSS_BETA):.0%}  (p=0.5 일 때)")

In [ ]:
def cls_heatmap(out, ci, b=0):
    s = torch.sigmoid(out["cls"][b]).view(FH, FW, NA)[..., ANCHOR_CLS[:NA] == ci]
    return s.max(-1).values.cpu()

def show_bev_pred(ax, pts, out, title, gt=None, thr=0.5, b=0):
    bev_ax(ax, pts, title, s=0.3)
    d = postprocess(out, b); k = d["scores"] > thr
    if gt is not None: draw_bev_boxes(ax, gt[0], color="white", lw=1.2, ls="--")
    draw_bev_boxes(ax, d["boxes"][k], color=["lime" if l == 0 else "cyan" for l in d["labels"][k].tolist()], lw=1.6,
                   texts=[f"{s:.2f}" for s in d["scores"][k].tolist()])
    return d

fig, axes = plt.subplots(1, 3, figsize=(27, 8.5))
for ax, ci in zip(axes[:2], range(2)):
    ax.imshow(cls_heatmap(out0, ci), origin="lower", extent=[XMIN, XMAX, YMIN, YMAX], cmap="jet", vmin=0, vmax=1)
    ax.set_title(f"untrained: '{CLASSES[ci]}' score map (max {float(cls_heatmap(out0, ci).max()):.3f})"); ax.set_aspect("equal")
show_bev_pred(axes[2], scene["points"], out0, "untrained: detections (score>0.5)", gt=(gtb.cpu(),))
plt.tight_layout(); plt.show()

## B6. 학습시키며 변화 관찰하기
합성 장면 `N_TRAIN` 개로 학습합니다 (좌우 반전 augmentation, mixed precision). 3D conv 때문에 실습 1 보다 무겁습니다.

In [ ]:
def flip_y(s):
    p = s["points"].copy(); p[:, 1] *= -1
    b = s["boxes"].copy(); b[:, 1] *= -1; b[:, 6] *= -1
    return dict(points=p, boxes=b, labels=s["labels"])

def to_batch(scenes):
    return ([torch.from_numpy(s["points"]).to(device) for s in scenes],
            [(torch.from_numpy(s["boxes"]).to(device), torch.from_numpy(s["labels"]).to(device)) for s in scenes])

snapshots, history = {0: INIT_STATE}, []
seed_all(); model = VoxelNet().to(device); model.load_state_dict(INIT_STATE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
it_per_ep = math.ceil(len(train_set) / BATCH_SIZE); total, warm = EPOCHS * it_per_ep, max(1, EPOCHS * it_per_ep // 20)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda it: (it + 1) / warm if it < warm else 0.5 * (1 + math.cos(math.pi * (it - warm) / max(1, total - warm))))
use_amp = device.type == "cuda"; scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    model.train(); order = np.random.permutation(len(train_set))
    for s_ in range(0, len(order), BATCH_SIZE):
        scenes = [flip_y(train_set[j]) if random.random() < 0.5 else train_set[j] for j in order[s_:s_ + BATCH_SIZE]]
        pts_b, gt_b = to_batch(scenes)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(pts_b)
        ld = compute_loss(out, gt_b); loss = sum(ld.values())
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0); scaler.step(opt); scaler.update(); sched.step()
        history.append({k: float(v.detach()) for k, v in ld.items()})
    msg = " ".join(f"{k}={np.mean([h[k] for h in history[-it_per_ep:]]):.3f}" for k in history[-1])
    print(f"epoch {ep:2d}/{EPOCHS} ({time.time() - t0:5.0f}s)  {msg}")
    if ep in SNAP_EPOCHS: snapshots[ep] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
del model
if use_amp: torch.cuda.empty_cache()

if history:
    fig, axes = plt.subplots(1, 3, figsize=(18, 3.5))
    for ax, k in zip(axes, history[0]):
        v = np.array([h[k] for h in history]); w = max(1, len(v) // 50)
        ax.plot(v, alpha=0.25, color="gray"); ax.plot(np.convolve(v, np.ones(w) / w, mode="valid"), color="tab:blue")
        ax.set_title(k); ax.set_xlabel("iteration")
    plt.tight_layout(); plt.show()

In [ ]:
model_snap = VoxelNet().to(device).eval()
def load_snap(ep): model_snap.load_state_dict(snapshots[ep]); return model_snap.eval()

eps = sorted(snapshots)
fig, axes = plt.subplots(len(eps), 3, figsize=(27, 8 * len(eps)))
for r_, ep in enumerate(eps):
    with torch.no_grad(): o = load_snap(ep)([pts_t])
    hm = cls_heatmap(o, 0)
    axes[r_, 0].imshow(hm, origin="lower", extent=[XMIN, XMAX, YMIN, YMAX], cmap="jet", vmin=0, vmax=1); axes[r_, 0].set_aspect("equal")
    axes[r_, 0].set_title(f"epoch {ep}: 'Car' score map (max {float(hm.max()):.2f})")
    bev_ax(axes[r_, 1], scene["points"], f"epoch {ep}: top-100 anchors decoded, before NMS", s=0.3)
    top = torch.sigmoid(o["cls"][0]).topk(100).indices
    draw_bev_boxes(axes[r_, 1], decode(o["box"][0][top], ANCHORS[top]), color="yellow", lw=0.6)
    show_bev_pred(axes[r_, 2], scene["points"], o, f"epoch {ep}: final detections (score>0.5)", gt=(gtb.cpu(),))
plt.tight_layout(); plt.show()

## B7. 정량 평가 — center-distance AP (nuScenes 방식)
예측 박스 중심이 GT 중심에서 **Car 2 m / Pedestrian 1 m** 이내면 정답으로 보고, score 순으로 매칭해 AP 를 계산합니다.
(KITTI 공식 지표는 회전 3D IoU 기반 AP 입니다.)

In [ ]:
@torch.no_grad()
def evaluate(model, scenes, thr_dist=(2.0, 1.0), score_thr=0.5):
    res = {c: dict(sc=[], tp=[], n_gt=0) for c in range(len(CLASSES))}
    for s in scenes:
        d = postprocess(model([torch.from_numpy(s["points"]).to(device)]))
        for c in range(len(CLASSES)):
            g = torch.from_numpy(s["boxes"][s["labels"] == c]).to(device); res[c]["n_gt"] += len(g)
            k = d["labels"] == c; db, ds = d["boxes"][k], d["scores"][k]
            used = torch.zeros(len(g), dtype=torch.bool, device=device)
            for j in ds.argsort(descending=True).tolist():
                ok = False
                if len(g):
                    dist = torch.norm(g[:, :2] - db[j, :2], dim=1); dist[used] = 1e9
                    m = int(dist.argmin()); ok = bool(dist[m] < thr_dist[c])
                    if ok: used[m] = True
                res[c]["sc"].append(float(ds[j])); res[c]["tp"].append(ok)
    out = {}
    for c, r in res.items():
        sc, tp = np.array(r["sc"]), np.array(r["tp"], dtype=float)
        o = np.argsort(-sc); tp = tp[o]; ctp = np.cumsum(tp); cfp = np.cumsum(1 - tp)
        rec = ctp / max(r["n_gt"], 1); prec = ctp / np.maximum(ctp + cfp, 1e-9)
        ap = float(np.sum(np.diff(np.r_[0, rec]) * np.maximum.accumulate(prec[::-1])[::-1])) if len(tp) else 0.0
        hi = sc[o] >= score_thr
        out[CLASSES[c]] = dict(AP=ap, P=float(tp[hi].mean()) if hi.any() else 0.0, R=float(tp[hi].sum() / max(r["n_gt"], 1)))
    return out

print(f"{'epoch':>5} | " + " | ".join(f"{c:^26s}" for c in CLASSES))
print(f"{'':>5} | " + " | ".join(f"{'AP':>8s}{'P@0.5':>9s}{'R@0.5':>9s}" for _ in CLASSES))
for ep in eps:
    r_ = evaluate(load_snap(ep), val_set)
    print(f"{ep:>5} | " + " | ".join(f"{r_[c]['AP']:8.3f}{r_[c]['P']:9.3f}{r_[c]['R']:9.3f}" for c in CLASSES))

In [ ]:
# 최종 모델의 3D 결과 (val 장면)
m_last = load_snap(eps[-1])
with torch.no_grad(): d = postprocess(m_last([pts_t]))
k = d["scores"] > 0.5
plot3d(scene["points"], [(scene["boxes"], "white", "GT"), (d["boxes"][k].cpu(), "lime", "prediction")],
       title=f"epoch {eps[-1]}: 3D detections on a validation scene")

> 🤔 **생각해보기**
> 1. epoch 이 지나면서 'Car' score map 과 최종 검출 결과가 어떻게 바뀌었나요?
> 2. 학습이 끝난 뒤에도 놓친 차량이 있다면, 그 차량은 가까이 있나요, 멀리 있나요? (A4 의 거리별 점 수 그래프를 떠올려 보세요.)
> 3. Car 와 Pedestrian 중 어느 쪽 AP 가 더 낮나요? 보행자는 차량보다 크기가 훨씬 작다는 점을 생각해 보세요.

**답변**

1. 학습 전(epoch 0)의 Car score map은 전체적으로 약 0.5의 거의 균일한 값(max 0.51)을 보이며, 최종 검출 박스도 GT 차량 위치와 관계없는 곳에 많이 나타난다. epoch 1에는 일부 차량 주변에 높은 점수가 형성되지만 배경에도 반응한다. epoch 3, 8, 15로 학습이 진행되면서 배경 점수는 낮아지고 차량 위치에 높은 점수가 더 좁게 집중된다. 최종 박스도 GT와 더 잘 겹친다. 다만 학습 후에도 오검출 박스가 남으므로 score map의 최대값이 1.00이라고 해서 모든 예측이 정확한 것은 아니다.

2. 시각화된 검증 장면(val_set[0])에서는 차량 GT 11대 모두에 대응하는 검출 박스가 있어, 놓친 차량은 관찰되지 않았다. 약 47 m 떨어진 차량들도 검출되었다. 따라서 이 장면에는 누락 차량의 근거리·원거리를 구분할 대상이 없다. A4에서 확인한 것처럼 멀리 있는 물체는 점 수가 줄어들기 때문에, 다른 장면에서는 원거리의 희소성이 미검출의 원인이 될 수 있다.

3. 최종 epoch 15의 AP는 Car 0.948(94.8%), Pedestrian 0.912(91.2%)로 보행자가 더 낮다. 보행자는 차량보다 작은 표면을 차지하여 같은 거리에서 얻는 점 수가 적고, 형상과 위치를 추정할 단서도 부족해 검출이 더 어렵다. 이 AP는 합성 검증 장면 60개에서 중심 거리로 판정한 실습 지표이며, 매칭 기준은 Car 2 m, Pedestrian 1 m이다. 보행자의 더 엄격한 거리 기준도 함께 고려해야 한다.


## B8. 합성 데이터로 학습한 모델을 실제 KITTI 프레임에 적용하면?
같은 센서 기하로 만든 데이터지만, 실제 데이터에는 시뮬레이터에 없는 것들(차량의 실제 형상, 유리창을 투과하는 레이저, 식생, 노면 경사, 다른 반사 특성 …)이 있습니다.
결과가 좋든 나쁘든, **훈련 데이터와 실제 데이터의 분포 차이(domain gap)** 를 직접 확인하는 실험입니다.

In [ ]:
kp = torch.from_numpy(P).to(device); kgt = torch.from_numpy(kitti["boxes"])
is_car = torch.tensor([n == "Car" for n in kitti["names"]])
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, ep in zip(axes, [0, eps[-1]]):
    with torch.no_grad(): o = load_snap(ep)([kp])
    d = show_bev_pred(ax, P, o, f"Real KITTI frame — epoch {ep} (white dashed = GT Car)", gt=(kgt[is_car],))
    ax.set_xlim(XMIN, min(XMAX, P[:, 0].max())); ax.plot(0, 0, "r^", ms=10)
plt.tight_layout(); plt.show()

with torch.no_grad(): d = postprocess(load_snap(eps[-1])([kp]))
k = d["scores"] > 0.5
im = kitti["image"]; fig, ax = plt.subplots(figsize=(18, 6)); ax.imshow(im); ax.axis("off")
for bx, cl in [(kgt[is_car], "white"), (d["boxes"][k].cpu(), "lime")]:
    for cn in box_corners_3d(bx).numpy():
        q, dpt = project(cn, kitti["lidar2img"])
        if (dpt > 0).all():
            for i, j in [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7)]:
                ax.plot(q[[i, j], 0], q[[i, j], 1], color=cl, lw=1.5)
ax.set_xlim(0, im.shape[1]); ax.set_ylim(im.shape[0], 0)
ax.set_title(f"Predictions (green, score>0.5: {int(k.sum())}) vs GT (white) projected onto the camera image"); plt.show()
plot3d(P, [(kgt, "white", "GT"), (d["boxes"][k].cpu(), "lime", "prediction")], title="Real KITTI frame — 3D")

## 정리

**핵심 요약**
- Point cloud 는 **순서 없는, 개수가 변하는 3D 점 집합**이고, 거리가 멀수록 점이 급격히 줄어듭니다.
- 같은 데이터라도 방법마다 입력이 다릅니다: 사람이 정한 BEV 점유 격자 (PIXOR) · **학습되는 voxel feature** (VoxelNet) · 원본 점 (PointRCNN).
- VoxelNet 은 ① voxel 로 묶고 ② VFE 로 점들의 feature 를 학습한 뒤 ③ 3D conv 로 높이 방향 정보를 모아 BEV 로 만들고, 이후는 **실습 1 과 같은 2D detection (RPN · anchor · NMS)** 으로 풉니다.
- 3D 에서 새로 생기는 것: 클래스별 실제 크기의 anchor, 7개 값의 회귀 (x, y, z, l, w, h, yaw).